1> Loading Word Docs

In [2]:
# libraries for reading word docs
from langchain_community.document_loaders import Docx2txtLoader,UnstructuredWordDocumentLoader

1a> Docx2txtLoader

In [10]:
try:
    docx_loader = Docx2txtLoader("data/word_files/notes.docx")
    docs = docx_loader.load()
    print(f"No of docs loaded: {len(docs)}")
    print(f"Content: {docs[0].page_content[:200]}")
    print(f"Metadata: {docs[0].metadata}")
except Exception as e:
    print(f"Error: {e}")

No of docs loaded: 1
Content: Unit 2 – RPC, Distributed Objects & Communication (Simplified Detailed Notes)

These notes explain every topic and subtopic from Unit 2 in a detailed yet easy-to-understand way. Difficult concepts are
Metadata: {'source': 'data/word_files/notes.docx'}


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
doc_content = docs[0].page_content
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size = 200,
    chunk_overlap = 20,
    length_function = len
)
doc_chunks = recursive_splitter.split_text(doc_content)
print(f"No of chunks created: {len(doc_chunks)}")
for i,chunk in enumerate(doc_chunks):
    print(f"Chunk{i+1}:")
    print(f"{chunk}")

No of chunks created: 72
Chunk1:
Chunk: Unit 2 – RPC, Distributed Objects & Communication (Simplified Detailed Notes)

These notes explain every topic and subtopic from Unit 2 in a detailed yet easy-to-understand way. Difficult concepts are
Chunk2:
Chunk: concepts are simplified using step-by-step explanations, real-life examples, and comparisons so they are easier to learn for exams.

Introduction to Remote Procedure Call (RPC)

Remote Procedure Call
Chunk3:
Chunk: Procedure Call (RPC) allows a program on one computer to execute a function on another computer as if the function were local.

Main Idea:
The programmer writes code normally, while RPC hides
Chunk4:
Chunk: while RPC hides networking complexity.

Example:
Suppose a food delivery app running on your phone needs restaurant data from another server.
Instead of manually handling:
- sockets
- packets
-
Chunk5:
Chunk: sockets
- packets
- network protocols

The app simply calls:
getRestaurantDetails()

The actual function runs on 

1b> UnstructuredWordDocumentLoader

In [6]:
try:
    unstructured_loader = UnstructuredWordDocumentLoader("data/word_files/notes.docx",mode="elements")
    unstructured_docs = unstructured_loader.load()
    print(f"No of elements loaded: {len(unstructured_docs)}")
    for i,doc in enumerate(unstructured_docs[:3]):
        print(f"Element: {i+1}")
        print(f"Content: {doc.page_content[:100]}")
        print(f"Metadata: {doc.metadata.get('category','unkown')}")
except Exception as e:
    print(f"Error: {e}")

No of elements loaded: 75
Element: 1
Content: Unit 2 – RPC, Distributed Objects & Communication (Simplified Detailed Notes)
Metadata: Title
Element: 2
Content: These notes explain every topic and subtopic from Unit 2 in a detailed yet easy-to-understand way. D
Metadata: NarrativeText
Element: 3
Content: Introduction to Remote Procedure Call (RPC)
Metadata: Title


SMART WORD DOC PROCESSOR

In [26]:
from typing import List
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
class SmartWordDocProcessor:

    def __init__(self,chunk_size=500,chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            separators=["\n\n","\n"," "],
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap
        )

    def process_worddoc(self,doc_path: str) -> List[Document]:
        # Loading Doc
        doc_loader = Docx2txtLoader(doc_path)
        doc_temp = doc_loader.load()

        # page content
        raw_chunks = self.text_splitter.split_text(doc_temp[0].page_content)

        cleaned_chunks=[]

        for i,data in enumerate(raw_chunks):
            temp = self._clean_data(data)
            # adding metadata
            cleaned_chunk = self.text_splitter.create_documents(
                texts=[temp],
                metadatas=[{
                    "chunk_no":i+1,
                    "total_chunks":len(raw_chunks),
                    "chunking_method":"smart_worddoc_processor",
                    "character_count": len(temp)
                }]
            )
            cleaned_chunks.extend(cleaned_chunk)
        
        return cleaned_chunks
    
    def _clean_data(self,text: str) -> str:
        text = " ".join(text.split())
        return text       

In [27]:
worddoc_processor = SmartWordDocProcessor()

In [33]:
# TESTING PROCESSOR
result = worddoc_processor.process_worddoc("data/word_files/notes.docx")
print(type(result))
print(f"No of chunks created: {len(result)}")
for i,chunk in enumerate(result[:3]):
    print(f"Chunk: {i+1}")
    print(f"Content: {chunk.page_content}")
    print("Metadata:")
    for key,value in chunk.metadata.items():
        print(f"{key}: {value}")

<class 'list'>
No of chunks created: 32
Chunk: 1
Content: Unit 2 – RPC, Distributed Objects & Communication (Simplified Detailed Notes) These notes explain every topic and subtopic from Unit 2 in a detailed yet easy-to-understand way. Difficult concepts are simplified using step-by-step explanations, real-life examples, and comparisons so they are easier to learn for exams. Introduction to Remote Procedure Call (RPC) Remote Procedure Call (RPC) allows a program on one computer to execute a function on another computer as if the function were local.
Metadata:
chunk_no: 1
total_chunks: 32
chunking_method: smart_worddoc_processor
character_count: 496
Chunk: 2
Content: Main Idea: The programmer writes code normally, while RPC hides networking complexity. Example: Suppose a food delivery app running on your phone needs restaurant data from another server. Instead of manually handling: - sockets - packets - network protocols The app simply calls: getRestaurantDetails() The actual function ru